# BGE-M3 Fine-Tune v2 — 10.6k pairs (4x previous dataset)

Previous run: 2,500 pairs → MRR@20 62.2% → 66.2% (+4.0%)
This run: 10,640 pairs, 6,245 unique anchors, 2,128 products — expect a larger lift.

All fixes from the previous round are baked in:
- `torchao` uninstalled (PEFT version clash)
- `LoraModel` tuner used directly (not `get_peft_model` which silently fails)
- Manual LoRA merge (sentence-transformers' `.fit()` discards the wrapper)
- Explicit tokenizer save (prevents missing `special_tokens_map.json`)

In [1]:
!pip install -q sentence-transformers faiss-cpu numpy peft
!pip uninstall -y -q torchao 2>/dev/null || true


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.6 MB/s eta 0:00:00


In [2]:
import os, importlib.util

# ---- EDIT THESE PATHS ----
GROUPED_TRAIN_JSON = "/kaggle/input/datasets/ameerhvmza/completedataset/grouped_training_data_v2.json"
HELD_OUT_EVAL_CSV  = "/kaggle/input/datasets/ameerhvmza/evaluationcomplete/held_out_eval_v2.csv"
FULL_CATALOG_CSV   = "/kaggle/input/datasets/ameerhvmza/allproducts/npk_prods_dmp.csv"

for name, path in [("TRAIN", GROUPED_TRAIN_JSON), ("EVAL", HELD_OUT_EVAL_CSV), ("CATALOG", FULL_CATALOG_CSV)]:
    ok = os.path.exists(path)
    print(f"  {name}: {'OK' if ok else 'MISSING'} -> {path}")
    if not ok:
        raise FileNotFoundError(f"{name} not found at {path}")

if importlib.util.find_spec("torchao") is not None:
    import torchao
    from packaging import version
    if version.parse(torchao.__version__) < version.parse("0.16.0"):
        raise ImportError(
            f"torchao {torchao.__version__} still installed. Restart kernel, re-run from top.")
    print(f"torchao {torchao.__version__} — OK")
else:
    print("torchao removed — clean.")


  TRAIN: OK -> /kaggle/input/datasets/ameerhvmza/completedataset/grouped_training_data_v2.json
  EVAL: OK -> /kaggle/input/datasets/ameerhvmza/evaluationcomplete/held_out_eval_v2.csv
  CATALOG: OK -> /kaggle/input/datasets/ameerhvmza/allproducts/npk_prods_dmp.csv
torchao removed — clean.


In [3]:
import pandas as pd
import numpy as np
import faiss

def evaluate(model, full_catalog_csv, held_out_eval_csv, top_k=20):
    full = pd.read_csv(full_catalog_csv, low_memory=False)
    full["text_to_embed"] = (
        full["name"].fillna("") + " " + full["brand"].fillna("") + " "
        + full["category_hierarchy"].fillna("")
    )
    product_names = full["name"].tolist()
    print(f"Embedding {len(product_names):,} catalogue products...")
    product_embeddings = model.encode(
        full["text_to_embed"].tolist(), convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True, batch_size=256)
    index = faiss.IndexFlatIP(product_embeddings.shape[1])
    index.add(np.asarray(product_embeddings, dtype="float32"))

    eval_df = pd.read_csv(held_out_eval_csv)
    results = []
    for _, row in eval_df.iterrows():
        q_emb = model.encode([row["query"]], convert_to_numpy=True, normalize_embeddings=True)
        _, indices = index.search(np.asarray(q_emb, dtype="float32"), top_k)
        rank = 0
        for i, idx in enumerate(indices[0]):
            if idx < len(product_names) and product_names[idx] == row["target_name"]:
                rank = i + 1
                break
        results.append({"query": row["query"], "target": row["target_name"], "rank": rank})

    df = pd.DataFrame(results)
    df["rr"] = df["rank"].apply(lambda r: 1/r if r > 0 else 0)
    mrr = df["rr"].mean() * 100
    hits = (df["rank"] > 0).mean() * 100
    print(f"  MRR@{top_k}: {mrr:.1f}%  |  Hit@{top_k}: {hits:.1f}%")
    misses = df[df["rank"] == 0]
    if len(misses):
        print(f"  {len(misses)}/{len(df)} queries missed target in top {top_k}")
    df.to_csv("heldout_results.csv", index=False)
    return df


In [4]:
import json, random
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import InputExample

class GroupedAnchorDataset(Dataset):
    def __init__(self, path):
        with open(path, encoding="utf-8") as f:
            self.anchor_to_positives = json.load(f)
        self.anchors = list(self.anchor_to_positives.keys())
        self._cursor = {a: 0 for a in self.anchors}
        n_pairs = sum(len(v) for v in self.anchor_to_positives.values())
        n_multi = sum(1 for v in self.anchor_to_positives.values() if len(v) > 1)
        print(f"Loaded {n_pairs:,} pairs -> {len(self.anchors):,} unique anchors "
              f"({n_multi:,} with >1 positive)")

    def set_epoch(self, epoch):
        random.Random(epoch).shuffle(self.anchors)

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, idx):
        anchor = self.anchors[idx]
        positives = self.anchor_to_positives[anchor]
        i = self._cursor[anchor] % len(positives)
        self._cursor[anchor] += 1
        return InputExample(texts=[anchor, positives[i]])

def make_dataloader(dataset, batch_size, epoch):
    dataset.set_epoch(epoch)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False)


In [5]:
import os, shutil

REQUIRED_FILES = [
    "config.json", "config_sentence_transformers.json", "model.safetensors",
    "modules.json", "sentence_bert_config.json", "tokenizer.json",
    "tokenizer_config.json", os.path.join("1_Pooling", "config.json"),
]

def verify_saved_model(path):
    missing = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(path, f))]
    if missing:
        raise RuntimeError(f"Save incomplete for '{path}'. Missing: {missing}")
    print(f"Verified: all {len(REQUIRED_FILES)} required files present in '{path}'.")

def merge_lora_weights(module):
    merged = 0
    for name, m in list(module.named_modules()):
        if hasattr(m, "base_layer") and hasattr(m, "merge") and callable(m.merge):
            m.merge()
            *parents, leaf = name.split(".")
            parent = module
            for p in parents:
                parent = getattr(parent, p)
            setattr(parent, leaf, m.base_layer)
            merged += 1
    if merged == 0:
        raise RuntimeError("merge_lora_weights found 0 LoRA layers to merge.")
    print(f"Merged {merged} LoRA layers into base weights.")
    return module


In [6]:
import torch
from sentence_transformers import SentenceTransformer, losses
from peft import LoraConfig, TaskType
from peft.tuners.lora import LoraModel

def train(config, dataset):
    print(f"\n{'='*60}")
    print(f"  Run {config['id']}: epochs={config['epochs']}  bs={config['batch_size']}  "
          f"lr={config['learning_rate']}  lora_r={config.get('lora_r', 8)}")
    print(f"{'='*60}\n")

    model = SentenceTransformer("BAAI/bge-m3")
    base_vocab_file = getattr(model.tokenizer, "vocab_file", None)

    # Discover module names
    attn_names = sorted({
        n.rsplit(".", 1)[-1] for n, _ in model[0].auto_model.named_modules()
        if any(k in n.lower() for k in ["query", "key", "value", "q_proj", "k_proj", "v_proj"])
    })
    print(f"Attention module names: {attn_names}")

    # Wrap with LoRA
    peft_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
        r=config.get("lora_r", 8), lora_alpha=32, lora_dropout=0.1,
        target_modules=["query", "key", "value", "q_proj", "k_proj", "v_proj"],
    )
    lora_model = LoraModel(model[0].auto_model, {"default": peft_config}, "default")
    model[0].auto_model = lora_model

    # Verify injection
    lora_params = [n for n, _ in model[0].auto_model.named_parameters() if "lora_" in n]
    if not lora_params:
        raise RuntimeError(f"No lora_ params found. Use these names: {attn_names}")
    trainable = sum(p.numel() for p in model[0].auto_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model[0].auto_model.parameters())
    print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
    print(f"LoRA tensors: {len(lora_params)}\n")

    # Train
    train_loss = losses.MultipleNegativesRankingLoss(model)
    for epoch in range(config["epochs"]):
        dl = make_dataloader(dataset, config["batch_size"], epoch)
        model.fit(
            train_objectives=[(dl, train_loss)], epochs=1,
            warmup_steps=10 if epoch == 0 else 0,
            optimizer_params={"lr": config["learning_rate"]}, use_amp=True)
    print("\nTraining complete.\n")

    # Verify LoRA survived .fit()
    auto = model[0].auto_model
    lora_after = {k: v for k, v in auto.state_dict().items() if "lora_" in k}
    print(f"After .fit(): {type(auto).__name__}, LoRA tensors: {len(lora_after)}")
    if not lora_after:
        raise RuntimeError("LoRA weights gone after .fit()")

    save_path = f"model_run_{config['id']}"

    # Adapter backup
    adapter_path = f"{save_path}_lora_adapter"
    os.makedirs(adapter_path, exist_ok=True)
    torch.save(lora_after, os.path.join(adapter_path, "adapter_model.bin"))
    peft_config.save_pretrained(adapter_path)
    print(f"Adapter backup: {len(lora_after)} tensors -> {adapter_path}/")

    # Merge + save
    merge_lora_weights(auto)
    model[0].auto_model = auto
    model.save(save_path)
    model.tokenizer.save_pretrained(save_path)

    if base_vocab_file and os.path.exists(base_vocab_file):
        dest = os.path.join(save_path, os.path.basename(base_vocab_file))
        if not os.path.exists(dest):
            shutil.copy(base_vocab_file, dest)

    verify_saved_model(save_path)
    shutil.make_archive(save_path, "zip", save_path)
    print(f"\nDone: {save_path}.zip")
    return model


/tmp/ipykernel_23/4022090339.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses


In [7]:
dataset = GroupedAnchorDataset(GROUPED_TRAIN_JSON)

BEST_CONFIG = {"id": 6, "epochs": 10, "batch_size": 8, "learning_rate": 5e-5, "lora_r": 32}

model = train(BEST_CONFIG, dataset)

evaluate(model, FULL_CATALOG_CSV, HELD_OUT_EVAL_CSV)


Loaded 10,390 pairs -> 6,133 unique anchors (1,207 with >1 positive)

  Run 6: epochs=10  bs=8  lr=5e-05  lora_r=32



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Verified: all 8 required files present in 'model_run_6'.

Done: model_run_6.zip
Embedding 155,444 catalogue products...


Batches:   0%|          | 0/608 [00:00<?, ?it/s]

  MRR@20: 32.9%  |  Hit@20: 64.4%
  89/250 queries missed target in top 20


,query,target,rank,rr
0,powder chilli,"Mehran Red Chilli Powder, 400g",1,1.000000
1,sabzi miks ready made,Food To Go Mix Vegetable Prepared In Olive Oil...,0,0.000000
2,chiken,D'Lish Chicken Makhani 250gm,0,0.000000
3,dlsh mrgh,D'Lish Chicken Makhani 250gm,1,1.000000
4,d lish ka murgh 250gm,D'Lish Chicken Makhani 250gm,1,1.000000
...,...,...,...,...
245,msala buti wala,"Shan Malai Boti Recipe Masala, 40g",1,1.000000
246,ka masaala 45g,"Falak Tikka Boti Masala, 45g",0,0.000000
247,masaala buti,"Falak Tikka Boti Masala, 45g",12,0.083333
248,masala buti ready made,"Falak Tikka Boti Masala, 45g",12,0.083333
